# Cell 1: Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import load_dataset

# Set visualization style
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (14, 6)

# Cell 2: Load "All" Available Splits

In [ ]:
data_frames = []

print("Attempting to load splits...")

for i in range(10):  # Try up to repliqa_9
    split_name = f"repliqa_{i}"
    try:
        # Load specific split
        ds = load_dataset("ServiceNow/repliqa", split=split_name)
        df_split = ds.to_pandas()

        # Add source column
        df_split["source_split"] = split_name
        data_frames.append(df_split)
        print(f"✅ Loaded {split_name}: {len(df_split)} rows")
    except Exception:
        # Stop trying if we hit a split that doesn't exist
        print(f"⏹️  Could not load {split_name} (likely doesn't exist yet). Stopping load.")
        break

# Combine and Process
if data_frames:
    df = pd.concat(data_frames, ignore_index=True)
    print(f"\nTotal raw records loaded: {len(df)}")

    # --- DATA PROCESSING STEPS ---

    # 1. Keep only the 4 required columns and rename 'long_answer' to 'context'
    cols_to_keep = ['document_topic', 'question_id', 'question', 'long_answer']
    df = df[cols_to_keep].rename(columns={'long_answer': 'context'})

    # 2. Remove rows with empty strings or NaN cells
    # We replace empty strings with pd.NA first, ensuring dropna() catches both types of missing data
    df = df.replace('', pd.NA).dropna()

    # 3. Filter for rows where context word count > 10
    # .str.split() defaults to splitting on whitespace, which is standard for counting words
    df = df[df['context'].str.split().str.len() > 10]

    # 4. Print the number of rows for each document_topic
    print(f"\nTotal records after cleaning and filtering: {len(df)}")
    print("\n--- Row count per Document Topic ---")
    print(df['document_topic'].value_counts().to_string())

else:
    print("❌ No data loaded.")

# Cell 3: Sampling, Train/Test Split, and KDE Visualization

In [ ]:
from sklearn.model_selection import train_test_split

# 1. Sample up to 2500 rows per document_topic
# The lambda function ensures we don't throw an error if a topic has < 2500 rows
df_sampled = df.groupby('document_topic', group_keys=False).apply(
    lambda x: x.sample(n=min(len(x), 2500), random_state=42)
).reset_index(drop=True)

print(f"Total rows after sampling: {len(df_sampled)}")

# Calculate the word count on the sampled dataframe before splitting
df_sampled['word_count'] = df_sampled['context'].str.split().str.len()

# 2. Train/Test Split (80/20)
# 'stratify' ensures the 80/20 proportion is maintained for EACH document_topic
train_df, test_df = train_test_split(
    df_sampled,
    test_size=0.20,
    random_state=42,
    stratify=df_sampled['document_topic']
)

print(f"Train set size: {len(train_df)}")
print(f"Test set size: {len(test_df)}\n")

# 3. Prepare data for the KDE Plot
# Add a label column so Seaborn knows how to color them
train_df = train_df.copy()
test_df = test_df.copy()
train_df['Dataset'] = 'Train (80%)'
test_df['Dataset'] = 'Test (20%)'

plot_df = pd.concat([train_df, test_df])

# 4. Plot KDE of word counts
plt.figure(figsize=(12, 6))
sns.kdeplot(
    data=plot_df,
    x='word_count',
    hue='Dataset',
    fill=True,
    common_norm=False,
    alpha=0.5,
    palette={'Train (80%)': '#1f77b4', 'Test (20%)': '#ff7f0e'}
)

plt.title('KDE Plot of Context Word Counts (Train vs Test Set Distribution)', fontsize=14)
plt.xlabel('Word Count', fontsize=12)
plt.ylabel('Density', fontsize=12)

# Limit x-axis to the 99th percentile to prevent extreme outliers from squishing the plot
max_x = plot_df['word_count'].quantile(0.99)
plt.xlim(0, max_x)

plt.tight_layout()
plt.show()

# Cell 4: Save Datasets to Google Drive

In [ ]:
import os

# 2. Define target paths
train_path = 'train_dataset.csv'
test_path = 'test_dataset.csv'

# 3. Ensure the target directory exists
output_dir = os.path.dirname(train_path)
os.makedirs(output_dir, exist_ok=True)

# 4. Save DataFrames to CSV
# Using index=False prevents pandas from writing the row numbers to the file
train_df.to_csv(train_path, index=False)
test_df.to_csv(test_path, index=False)

print(f"✅ Train dataset successfully saved to: {train_path}")
print(f"✅ Test dataset successfully saved to: {test_path}")